The notebook is divided in these parts:
## 1.   *Importing Libraries*
## 2.   *Importing Dataset*
## 3.   *Preprocessing*
## 4.   *Feature Engineering*
## 5.   *Model Training*
## 6.   *Evaluation and Prediction*
## 7.   *Deployment*
---

# 📔 Project Notebook Structure
```bibtex
1. Tasks are distributed across teams and organized by notebook titles
   Each notebook documents:
    - Code implementation for task completion
    - Relevant comments and annotations

2. Agile Project Methodology:
    Weekly iterations training 1-2 models

    Project completion criteria:
        Best performing model achieves target evaluation metrics

3. The Pipeline of notebook:
   a. Doing EDA on dataset and finding information about that
   
   b. Applying preprocessing on dataset by information:
      i.   Class balancing
      ii.  Data augmentation
      iii. Normalization
      iv.  De-noising

   c. Feature extraction:
      i. Using some trained model
      ii. Extracting using Deep Learning (MFCC, MEL spectograms)

   d. Model training:
      i. Using trained model and fine tune that
      ii. Implement from scratch with torch, keras

   e. Evaluation with accuracy, precision, recall, f1-score (Confusion Matrix)
```

💡 Tips:
``` python
  1. For unfinished section use this: #TODO
  2. The structure of task like this:
      # [TASK NAME] (Assignee: @username)
      # Sprint Goal: [e.g., "Improve Wav2Vec2 accuracy by 5%"]
      # Deadline: DD/MM  
```


# 1. *Libraries*



**Installing**



In [43]:
!pip install numpy==2.0.0 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 47.3 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.1.3
    Uninstalling numpy-2.1.3:
      Successfully uninstalled numpy-2.1.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.0.0 which is incompatible.
hazm 0.10.0 requires numpy==1.24.3, but you have numpy 2.0.0 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which is incompatible.
tensorflow-text 2.18.1 requires tensorflow<2.19,>=2.18.0, but you have tensorflow 2.19.0 which is incompatible.
tensorflow-decision-forests 1.11.0 requires tensorflow==2.18.0, but you have tensorflow 2.19.0 which is incompatible.
tf-keras 2.18.0 requir

In [1]:
!pip install openai-whisper -q -U

**Importing**

In [2]:
# For Data Processing
import numpy as np
import pandas as pd
import os
import shutil
import random
from sklearn.utils import shuffle
from tqdm import tqdm
from pathlib import Path

#  For Plotting
import matplotlib.pyplot as plt
import seaborn as sns


# For Speech Processing
import librosa
import scipy.signal as signal
import IPython.display as ipd
import soundfile as sf
import noisereduce as nr

# For DL, Classifier,... Model (Sklearn, pytorch, keras, Tensorflow,...)
import tensorflow_hub as hub

# TODO: need to be fixed by version
import whisper
# import openl3

import torchaudio
from tensorflow.keras.utils import to_categorical
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from sklearn.model_selection import train_test_split

# If you need other libraries to be imported write it here
import kagglehub
from abc import ABC, abstractmethod
from dataclasses import dataclass


# *2. Dataset - EDA*

Importing Dataset - (Assignee: Sepehr)\
Sprint Goal: [Importing Dataset from kaggle and making a folder in colab]\
Deadline: 02/May

In [21]:
# Importing Dataset From Kaggle
path = kagglehub.dataset_download("sepehrsimkhah/speech-dataset-pazhvak")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/speech-dataset-pazhvak


In [5]:
# Validating Dataset Path
list_of_dataset = os.listdir(path)
list_of_dataset

['1601-2000',
 '2401-2800',
 '401-800',
 '1-400',
 '801-1200',
 '3601-4018',
 '3201-3600',
 '2801-3200',
 '1201-1600',
 '2001-2400',
 'P1993818924117826_1247741184149835.xlsx']

In [8]:
word_table = os.path.join(path, 'P1993818924117826_1247741184149835.xlsx')
df = pd.read_excel(word_table)
df.sample(5)

,Folder Number,Persian Word
2493,2494,بداری
2430,2431,خورده‌ایم
939,940,احسان
1236,1237,گشته اید
3128,3129,معتاد


In [91]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4018 entries, 0 to 4017
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Folder Number  4018 non-null   int64 
 1   Persian Word   4017 non-null   object
dtypes: int64(1), object(1)
memory usage: 62.9+ KB


In [92]:
df.isna()

,Folder Number,Persian Word
0,False,False
1,False,False
2,False,False
3,False,False
4,False,False
...,...,...
4013,False,False
4014,False,False
4015,False,False
4016,False,False


## TODO: Fixing the voice problem (assignee: Sepehr)

In [95]:
# by doing this we found out that we have a nan value
df_false = df[df.isna().any(axis=1)]
df_false

,Folder Number,Persian Word
662,663,NaN


i listend the voices of folder 663th and i found out that the 663 is "کارفرما"

In [96]:
row = df[df["Persian Word"] == "کارفرما"]
row

,Folder Number,Persian Word
3778,3779,کارفرما


this word has two folders, I Don't know why but:\
a folder is "کارفرما" with men voices (3779)\
the other one is "کارفرما" with women voices (663)

In [9]:
# it's 10 folders
list_of_folders = list_of_dataset.copy()
list_of_folders.remove('P1993818924117826_1247741184149835.xlsx')
list_of_folders

['1601-2000',
 '2401-2800',
 '401-800',
 '1-400',
 '801-1200',
 '3601-4018',
 '3201-3600',
 '2801-3200',
 '1201-1600',
 '2001-2400']

In [79]:
# finding the audio of word by number folder

def find_word_range(folder_number:int, folders:list):
  for word_range in folders:
    start_word_range, stop_word_range = word_range.split("-")
    if int(start_word_range) <= folder_number <= int(stop_word_range):
      return start_word_range, stop_word_range, word_range

random_number_folder = random.randint(0, 4018)
word = (df[df["Folder Number"] == random_number_folder])["Persian Word"].values[0]
_, _, word_range = find_word_range(random_number_folder, list_of_folders)
path_example = os.path.join(f"{path}/{word_range}/{random_number_folder}/")
list_audio_example = os.listdir(path_example)
audio_file = path_example + list_audio_example[random.randint(0, len(list_audio_example)-1)]

print(f"Word: {word}")
print("Loading:", audio_file)
assert os.path.exists(audio_file), f"File {audio_file} not found!"

y, sr = librosa.load(audio_file, sr=None)
print(sr)
ipd.Audio(y, rate=sr)

# این شستن خیلی باحال گفته به همین خاطر دستش نزنید🙂

Word: 204    شستن
Name: Persian Word, dtype: object
Loading: /kaggle/input/speech-dataset-pazhvak/1-400/205/FA_16000_0013.wav
16000


In [80]:
# Some Examples

def play_audio_samples(path:str, word_table:pd.DataFrame, folders:list, num_samples:int=2):
    for _ in range(num_samples):
        random_number_folder = random.randint(0, len(word_table) - 1)

        word_row = word_table[word_table["Folder Number"] == random_number_folder]
        if word_row.empty:
            continue

        word = word_row["Persian Word"].values[0]
        start_range, end_range, word_range = find_word_range(random_number_folder, folders)

        folder_path = os.path.join(path, word_range, str(random_number_folder))
        if not os.path.exists(folder_path):
            print(f"Folder not found: {folder_path}")
            continue

        audio_files = os.listdir(folder_path)
        if not audio_files:
            print(f"No audio files in: {folder_path}")
            continue

        audio_file = os.path.join(folder_path, random.choice(audio_files))

        print(f"\nWord: {word} (Folder: {random_number_folder})")
        print(f"Audio file: {audio_file}")

        try:
            y, sr = librosa.load(audio_file, sr=None)
            display(ipd.Audio(y, rate=sr))
        except Exception as e:
            print(f"Error loading audio: {e}")

In [88]:
play_audio_samples(path=path,word_table=df,folders=list_of_folders,num_samples=3)


Word: شنید  (Folder: 120)
Audio file: /kaggle/input/speech-dataset-pazhvak/1-400/120/FA_16000_0008.wav



Word: سوراخ (Folder: 591)
Audio file: /kaggle/input/speech-dataset-pazhvak/401-800/591/FA_16000_0021.wav



Word: زمین شناس (Folder: 3915)
Audio file: /kaggle/input/speech-dataset-pazhvak/3601-4018/3915/FA_16000_0015.wav


EDA On Dataset - (Assignee: Sepehr)\
Sprint Goal: [Using some functions to analyzing voices and getting some informations]\
Deadline: 02/May

In [ ]:
# class to return dataset information

class PazhvakDataset:
    """A class to handle Persian audio dataset with metadata and folder structure."""

    def __init__(self, dataset_path: str, metadata_file: str):
        """
        Args:
            dataset_path (str): Root path of the dataset
            metadata_file (str): Path to metadata file (Excel/CSV) relative to dataset_path
        """
        self.dataset_path = dataset_path
        self.metadata_path = os.path.join(dataset_path, metadata_file)

        # Load metadata
        self.metadata = self._load_metadata()
        self.word_range_folders = self._extract_word_ranges(metadata_file)

        # Initialize splits
        self.train_indices: list = []
        self.val_indices: list = []
        self.test_indices: list = []

    def _load_metadata(self) -> pd.DataFrame:
        """Load metadata file (supports .csv, .xlsx)"""
        if self.metadata_path.endswith('.csv'):
            return pd.read_csv(self.metadata_path)

        elif self.metadata_path.endswith('.xlsx'):
            return pd.read_excel(self.metadata_path)

        else:
            raise ValueError("Unsupported file format. Use .csv or .xlsx")

    def _extract_word_ranges(self, metadata_file: str) -> list:
        """Extract folder ranges from dataset path"""
        list_of_folders = os.listdir(self.dataset_path)
        return list_of_folders.remove(metadata_file)

    def __len__(self) -> int:
        """Returns total number of audio folders"""
        return len(self.word_range_folders)

    def get_audio_sample(self, index: int = None, num_samples: int = 1) -> list:
        """
        Get audio sample(s) by index or random

        Args:
            index (int): Specific index to get (None for random)
            num_samples (int): Number of samples to return (default: 1)

        Returns:
            list: List of tuples (audio_data, sample_rate, word, metadata_row)
        """
        if num_samples < 1:
            raise ValueError("num_samples must be at least 1")

        samples = []

        for _ in range(num_samples):
            current_index = index if index is not None else random.randint(0, len(self.metadata)-1)

            try:
                row = self.metadata.iloc[current_index]
                folder_num = row['Folder Number']
                word_range = self._find_word_range(folder_num)

                audio_folder = os.path.join(
                    self.dataset_path,
                    word_range,
                    str(folder_num))

                if not os.path.exists(audio_folder):
                    continue

                audio_files = [f for f in os.listdir(audio_folder) if f.endswith(('.wav', '.mp3', '.flac'))]
                if not audio_files:
                    continue

                audio_file = os.path.join(audio_folder, random.choice(audio_files))
                y, sr = librosa.load(audio_file, sr=None)

                samples.append((y, sr, row['Persian Word'], row))
            except Exception as e:
                print(f"Error loading sample {current_index}: {str(e)}")
                continue

        return samples

    def _find_word_range(self, folder_number: int) -> str:
        """Find which range folder the number belongs to"""
        for word_range in self.word_range_folders:
            start, end = map(int, word_range.split('-'))
            if start <= folder_number <= end:
                return word_range
        raise ValueError(f"Folder number {folder_number} out of range")

    def split_dataset(self, test_size: float = 0.2, val_size: float = 0.1,
                     random_state: int = 42) -> None:
        """
        Split dataset into train/val/test sets

        Args:
            test_size (float): Proportion for test set
            val_size (float): Proportion for validation set
            random_state (int): Random seed for reproducibility
        """
        train_val_idx, test_idx = train_test_split(
            range(len(self.metadata)),
            test_size=test_size,
            random_state=random_state
        )

        train_idx, val_idx = train_test_split(
            train_val_idx,
            test_size=val_size/(1-test_size),
            random_state=random_state
        )

        self.train_indices = train_idx
        self.val_indices = val_idx
        self.test_indices = test_idx

    def play_random_sample(self, split: str = None, num_samples: int = 1) -> None:
        """
        Play random audio samples from specified split

        Args:
            split (str): One of 'train', 'val', 'test', or None for entire dataset
            num_samples (int): Number of samples to play (default: 1)
        """
        if split not in [None, 'train', 'val', 'test']:
            raise ValueError("split must be one of: None, 'train', 'val', 'test'")

        if num_samples < 1:
            raise ValueError("num_samples must be at least 1")

        if split and not getattr(self, f"{split}_indices"):
            raise ValueError(f"{split} split not initialized. Call split_dataset() first")

        for i in range(num_samples):
            index = None
            if split == 'train':
                index = random.choice(self.train_indices)
            elif split == 'val':
                index = random.choice(self.val_indices)
            elif split == 'test':
                index = random.choice(self.test_indices)
            else:
                index = random.randint(0, len(self.metadata)-1)

            try:
                y, sr, word, metadata = self.get_audio_sample(index)[0]  # Get first (and only) sample
                print(f"\nSample #{i+1}/{num_samples}")
                print(f"Word: {word}")
                print(f"Folder: {metadata['Folder Number']}")
                print(f"Audio length: {len(y)/sr:.2f} seconds")
                display(ipd.Audio(y, rate=sr))
            except Exception as e:
                print(f"Error playing sample: {str(e)}")
                continue

In [1]:
dataset = PazhvakDataset(dataset_path=path,metadata_file='P1993818924117826_1247741184149835.xlsx')

# # Split dataset
# dataset.split_dataset(test_size=0.2, val_size=0.1)


single_sample = dataset.get_audio_sample(index=42)
four_samples = dataset.get_audio_sample(num_samples=4)
dataset.play_random_sample()

NameError: name 'PazhvakDataset' is not defined

Exploratory Data Analysis

In [ ]:
class IAudioAnalyzer(ABC):
    @abstractmethod
    def analyze(self, signal: np.ndarray, sample_rate: int) -> dict:
        pass

    @abstractmethod
    def plot(self, signal: np.ndarray, sample_rate: int):
        pass

#-------------------------------------------------------------
#<<<<<<<<<<<<<<<<<<<Audio Analysis>>>>>>>>>>>>>>>>>>>>>>>>>>>>
#-------------------------------------------------------------

# functions to analyzing voices

def raw_waveform_plot(signal:list, sample_rate:int=16000):
    librosa.display.waveshow(y=signal, sr=sample_rate)

def fft_spectrum_plot(signal:list, sample_rate:int=16000):
    fft = np.fft.fft(signal)
    freq = np.fft.fftfreq(len(signal), 1/sample_rate)
    plt.plot(freq[:len(freq)//2], np.abs(fft)[:len(freq)//2])

def stft_spectrograms_plot(signal:list, sample_rate:int=16000, scale:str="linear"):
    D = librosa.stft(signal, n_fft=2048, hop_length=512)
    s_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)
    librosa.display.specshow(s_db, sr=sample_rate, hop_length=512, x_axis='time', y_axis=scale)
    plt.colorbar(format='%+2.0f dB')

def mel_plot(signal:list, sample_rate:int=16000, n_mels:int=80):
    S = librosa.feature.melspectrogram(y=signal, sr=sample_rate, n_mels=n_mels)
    S_db = librosa.power_to_db(S, ref=np.max)
    return S_db

def mfcc_plot(signal:list, sample_rate:int=16000, n_mfcc:int=13):
    mfccs = librosa.feature.mfcc(y=signal, sr=sample_rate, n_mfcc=n_mfcc)
    librosa.display.specshow(mfccs, x_axis="time")
    plt.colorbar()

def word_count(self) -> pd.DataFrame:
    """
    Count voices for each word and return summary DataFrame

    Returns:
        DataFrame with columns: ['Persian Word', 'Voice Count', 'Folder Count']
    """
    word_stats = []

    for word, group in self.metadata.groupby('Persian Word'):
        voice_count = 0
        folder_count = len(group)

        for _, row in group.iterrows():
            folder_num = row['Folder Number']
            word_range = self._find_word_range(folder_num)
            folder_path = os.path.join(self.dataset_path, word_range, str(folder_num))

            if os.path.exists(folder_path):
                voice_count += len([f for f in os.listdir(folder_path)
                                 if f.endswith(('.wav', '.mp3', '.flac'))])

        word_stats.append({
            'Persian Word': word,
            'Voice Count': voice_count,
            'Folder Count': folder_count
        })

    return pd.DataFrame(word_stats
def most_word_count(self, top_n: int = 10) -> pd.DataFrame:
    """
    Show words with most voices (top N)

    Args:
        top_n: Number of top results to show
    """
    word_stats = self.word_count()
    sorted_stats = word_stats.sort_values('Voice Count', ascending=False)

    print(f"\nTop {top_n} words by voice count:")
    print(sorted_stats.head(top_n))

    return sorted_stats.head(top_n
def less_word_count(self, top_n: int = 10) -> pd.DataFrame:
    """
    Show words with least voices (top N)

    Args:
        top_n: Number of bottom results to show
    """
    word_stats = self.word_count()
    sorted_stats = word_stats.sort_values('Voice Count')

    print(f"\nBottom {top_n} words by voice count:")
    print(sorted_stats.head(top_n))

    return sorted_stats.head(top_n
def plot_voice_distribution(self):
    """Plot distribution of voice counts per word"""
    word_stats = self.word_count()
    voice_counts = word_stats['Voice Count'].value_counts().sort_index()

    plt.figure(figsize=(12, 6))
    voice_counts.plot(kind='bar')
    plt.title("Distribution of Voice Counts per Word")
    plt.xlabel("Number of Voices")
    plt.ylabel("Number of Words")
    plt.grid(True)
    plt.show()

    # Print top 10 voice counts
    print("\nVoice counts distribution:")
    for count, num_words in voice_counts.sort_index(ascending=False).head(10).items():
        print(f"Voices with count {count} voices: {num_words} words"
def analyze_durations(self) -> dict:
    """
    Analyze audio durations across all voices

    Returns:
        dict: {'avg': float, 'max': float, 'min': float,
              'above_avg': int, 'below_avg': int}
    """
    durations = []
    total_files = 0

    # Collect durations with progress bar
    for _, row in tqdm(self.metadata.iterrows(), total=len(self.metadata), desc="Analyzing durations"):
        folder_num = row['Folder Number']
        word_range = self._find_word_range(folder_num)
        folder_path = os.path.join(self.dataset_path, word_range, str(folder_num))

        if os.path.exists(folder_path):
            audio_files = [f for f in os.listdir(folder_path)
                         if f.endswith(('.wav', '.mp3', '.flac'))]
            total_files += len(audio_files)

            for file in audio_files:
                try:
                    y, sr = librosa.load(os.path.join(folder_path, file), sr=None)
                    durations.append(len(y)/sr)
                except:
                    continue

    if not durations:
        return {}

    avg_dur = sum(durations)/len(durations)
    max_dur = max(durations)
    min_dur = min(durations)

    above_avg = sum(1 for d in durations if d > avg_dur)
    below_avg = len(durations) - above_avg

    print(f"\nDuration Analysis:")
    print(f"Total voices analyzed: {total_files}")
    print(f"Average duration: {avg_dur:.2f} seconds")
    print(f"Max duration: {max_dur:.2f} seconds")
    print(f"Min duration: {min_dur:.2f} seconds")
    print(f"Voices above average: {above_avg} ({above_avg/total_files:.1%})")
    print(f"Voices below average: {below_avg} ({below_avg/total_files:.1%})")

    return {
        'avg': avg_dur,
        'max': max_dur,
        'min': min_dur,
        'above_avg': above_avg,
        'below_avg': below_avg}

def plot_duration_distribution(self):
    """Plot histogram of audio durations"""
    durations = []

    # Sample some files for faster plotting
    sample_size = min(500, len(self.metadata))
    sampled_metadata = self.metadata.sample(sample_size)

    for _, row in tqdm(sampled_metadata.iterrows(), total=sample_size, desc="Sampling durations"):
        folder_num = row['Folder Number']
        word_range = self._find_word_range(folder_num)
        folder_path = os.path.join(self.dataset_path, word_range, str(folder_num))

        if os.path.exists(folder_path):
            audio_files = [f for f in os.listdir(folder_path)
                         if f.endswith(('.wav', '.mp3', '.flac'))]
            for file in audio_files[:2]:  # Take max 2 files per folder
                try:
                    y, sr = librosa.load(os.path.join(folder_path, file), sr=None)
                    durations.append(len(y)/sr)
                except:
                    continue

    plt.figure(figsize=(12, 6))
    plt.hist(durations, bins=50, edgecolor='black')
    plt.title("Distribution of Audio Durations")
    plt.xlabel("Duration (seconds)")
    plt.ylabel("Number of Voices")
    plt.grid(True)
    plt.show()

In [ ]:
#-------------------------------------------------------------
#<<<<<<<<<<<<<<<<<<<<Word Analysis>>>>>>>>>>>>>>>>>>>>>>>>>>>>
#-------------------------------------------------------------

In [ ]:
#-------------------------------------------------------------
#<<<<<<<<<<<<<<<<<<<Dataset Statistics>>>>>>>>>>>>>>>>>>>>>>>>
#-------------------------------------------------------------

📝 **RESULT:**



> Dataset Information:


```
Dataset Problem:

Requirements:

```





# *3. Preprocessing*

Writing Preprocessing Functions - (Assignee: Mohammad)\
Sprint Goal: [Writing important functions for speech preprocessing]\
Deadline: 02/May

In [ ]:
#TODO: Writing each function in a cell with example running

# *4. Feature Extraction*

Writing Feature Extraction Functions - (Assignee: Shakiba)\
Sprint Goal: [Writing function for extracting features from voices for different model training]\
Deadline: 02/May

In [ ]:
#TODO: Writing each feature extraction function in a cell with example (maybe some function need some preprocessing, if it is please comment on that)

# *5. Model Training*

# *6. Evaluation and Prediction*

# *7. Deployment*

# *8. Model no.1*

Testing Trained Model - (Assignee: Sepehr)\
Sprint Goal: [Importing a trained persian model to test dataset how good is it]\
Deadline: 02/May

In [ ]:
model_name = "m3hrdadfi/wav2vec2-large-xlsr-persian"
processor = Wav2Vec2Processor.from_pretrained(model_name)
model = Wav2Vec2ForCTC.from_pretrained(model_name)

waveform, sample_rate = torchaudio.load("your_audio.wav")
resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
waveform = resampler(waveform)

inputs = processor(waveform.squeeze(), sampling_rate=16000, return_tensors="pt", padding=True)

with torch.no_grad():
    logits = model(**inputs).logits

predicted_ids = torch.argmax(logits, dim=-1)
transcription = processor.batch_decode(predicted_ids)[0]

print("Transcription:", transcription)


In [ ]:
model = whisper.load_model("base")
result = model.transcribe("your_audio.wav", language="fa")
transcription = result["text"]

print("Transcription:", transcription)
